In [28]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [29]:
import torch
import torch.nn.functional as F
import numpy as np
import copy
import matplotlib.pyplot as plt
%matplotlib inline

import allknighter
torch.manual_seed(42)


In [30]:
class ValueNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Flatten(),

            torch.nn.Linear(12*8*8, 256),
            torch.nn.BatchNorm1d(256),
            torch.nn.ReLU(),

            torch.nn.Linear(256, 64),
            torch.nn.BatchNorm1d(64),
            torch.nn.ReLU(),

            torch.nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)


In [31]:
def generate_game(max_moves=100):
    pieces = copy.deepcopy(allknighter.pieces)
    game_positions = []
    color = 1

    for _ in range(max_moves):
        board = allknighter.make_board(pieces)
        legal_moves = []

        for x,y,p,c in pieces:
            if c != color:
                continue

            if p == 1:
                moves = allknighter.pawn(board,[x,y,p,c])
            elif p == 3:
                moves = allknighter.knight(board,[x,y,p,c])
            elif p == 4:
                moves = allknighter.bishop(board,[x,y,p,c])
            elif p == 5:
                moves = allknighter.rook(board,[x,y,p,c])
            elif p == 9:
                moves = allknighter.queen(board,[x,y,p,c])
            elif p == 10:
                moves = allknighter.king(board,[x,y,p,c])
            else:
                continue

            for mx,my in moves:
                legal_moves.append((x,y,mx,my))

        if not legal_moves:
            break

        game_positions.append(copy.deepcopy(pieces))

        x,y,mx,my = legal_moves[np.random.randint(len(legal_moves))]

        pieces = [p for p in pieces if not (p[0]==mx and p[1]==my)]
        for p in pieces:
            if p[0]==x and p[1]==y:
                p[0],p[1] = mx,my
                break

        color *= -1

    return game_positions

[[[0, 0, 5, 1],
  [1, 0, 3, 1],
  [2, 0, 4, 1],
  [3, 0, 9, 1],
  [4, 0, 10, 1],
  [5, 0, 4, 1],
  [6, 0, 3, 1],
  [7, 0, 5, 1],
  [0, 1, 1, 1],
  [1, 1, 1, 1],
  [2, 1, 1, 1],
  [3, 1, 1, 1],
  [4, 1, 1, 1],
  [5, 1, 1, 1],
  [6, 1, 1, 1],
  [7, 1, 1, 1],
  [0, 7, 5, -1],
  [1, 7, 3, -1],
  [2, 7, 4, -1],
  [3, 7, 9, -1],
  [4, 7, 10, -1],
  [5, 7, 4, -1],
  [6, 7, 3, -1],
  [7, 7, 5, -1],
  [0, 6, 1, -1],
  [1, 6, 1, -1],
  [2, 6, 1, -1],
  [3, 6, 1, -1],
  [4, 6, 1, -1],
  [5, 6, 1, -1],
  [6, 6, 1, -1],
  [7, 6, 1, -1]],
 [[0, 0, 5, 1],
  [1, 0, 3, 1],
  [2, 0, 4, 1],
  [3, 0, 9, 1],
  [4, 0, 10, 1],
  [5, 0, 4, 1],
  [6, 0, 3, 1],
  [7, 0, 5, 1],
  [0, 1, 1, 1],
  [1, 1, 1, 1],
  [2, 1, 1, 1],
  [3, 1, 1, 1],
  [4, 3, 1, 1],
  [5, 1, 1, 1],
  [6, 1, 1, 1],
  [7, 1, 1, 1],
  [0, 7, 5, -1],
  [1, 7, 3, -1],
  [2, 7, 4, -1],
  [3, 7, 9, -1],
  [4, 7, 10, -1],
  [5, 7, 4, -1],
  [6, 7, 3, -1],
  [7, 7, 5, -1],
  [0, 6, 1, -1],
  [1, 6, 1, -1],
  [2, 6, 1, -1],
  [3, 6, 1, -1],
  [4, 6

In [32]:
def extract_move(pos_a, pos_b):
    set_a = {(x,y,p,c) for x,y,p,c in pos_a}
    set_b = {(x,y,p,c) for x,y,p,c in pos_b}

    removed = set_a - set_b
    added   = set_b - set_a

    if len(removed) == 1 and len(added) == 1:
        # normal move
        x1,y1,p,c = removed.pop()
        x2,y2,_,_ = added.pop()
        return (x1,y1,x2,y2,p,c)

    if len(removed) == 2 and len(added) == 1:
        # capture
        added_piece = added.pop()
        mover = None
        for piece in removed:
            if piece[3] == added_piece[3]:
                mover = piece
        x1,y1,p,c = mover
        x2,y2,_,_ = added_piece
        return (x1,y1,x2,y2,p,c)

    return None

game1=generate_game()
def moves(r):
    moves1=[]
    for i in range(len(r)-1):
        moves1.append(extract_move(r[i], r[i + 1]))
    return moves1


def evaluate(r):
    for i, pos in enumerate(r):
        print(i, allknighter.score(pos))
evaluate(game1)


0 0
1 0
2 0
3 0
4 0
5 0
6 0
7 0
8 0
9 0
10 0
11 0
12 -3
13 -3
14 -3
15 -3
16 -3
17 -3
18 -3
19 -3
20 -3
21 -3
22 -3
23 -3
24 -3
25 -3
26 -3
27 -3
28 -4
29 5
30 5
31 5
32 5
33 5
34 5
35 6
36 6
37 6
38 6
39 6
40 6
41 7
42 7
43 7
44 7
45 10
46 10
47 10
48 10
49 10
50 -990
51 -989
52 -989
53 -989
54 -989
55 -989
56 -990
57 -990
58 -990
59 -990
60 -991
61 -991
62 -991
63 -991
64 -991
65 -991
66 -991
67 -990
68 -995
69 -995
70 -995
71 -995
72 -995
73 -994
74 -994
75 -994
76 -994
77 -993
78 -993
79 -993
80 -993
81 -993
82 -993
83 -992
84 -992
85 -992
86 -992
87 -992
88 -992
89 -992
90 -992
91 -992
92 -992
93 -992
94 -992
95 -992
96 -992
97 -992
98 -992
99 -992


In [33]:
def encode_position(pieces):
    board = np.zeros((12, 8, 8), dtype=np.float32)

    piece_to_plane = {
        (1,  1): 0,  # white pawn
        (3,  1): 1,  # white knight
        (4,  1): 2,  # white bishop
        (5,  1): 3,  # white rook
        (9,  1): 4,  # white queen
        (10, 1): 5,  # white king
        (1, -1): 6,  # black pawn
        (3, -1): 7,
        (4, -1): 8,
        (5, -1): 9,
        (9, -1): 10,
        (10,-1): 11,
    }

    for x, y, p, c in pieces:
        plane = piece_to_plane[(p, c)]
        board[plane, y, x] = 1.0

    return board


In [34]:
def build_dataset(n_games=200):
    X = []
    y = []

    for _ in range(n_games):
        game = generate_game()
        for pos in game:
            X.append(encode_position(pos))
            y.append(allknighter.score(pos))

    X = torch.tensor(np.array(X))
    y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    return X, y

X,y=build_dataset(200)
print(X.shape,y.shape)

torch.Size([20000, 12, 8, 8]) torch.Size([20000, 1])


In [35]:
model = ValueNet()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = torch.nn.MSELoss()

In [36]:
batch_size = 256
epochs = 50
losses = []

for epoch in range(epochs):
    perm = torch.randperm(len(X))
    total_loss = 0

    for i in range(0, len(X), batch_size):
        idx = perm[i:i+batch_size]
        xb = X[idx]
        yb = y[idx]

        optimizer.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / (len(X) // batch_size)
    losses.append(avg_loss)
    print(f"Epoch {epoch}: loss = {avg_loss:.3f}")

Epoch 0: loss = 155497.987
Epoch 1: loss = 152999.273
Epoch 2: loss = 147237.996
Epoch 3: loss = 140875.223
Epoch 4: loss = 133284.325
Epoch 5: loss = 126306.778
Epoch 6: loss = 117602.591
Epoch 7: loss = 109711.301
Epoch 8: loss = 100095.893
Epoch 9: loss = 91879.987
Epoch 10: loss = 81931.244
Epoch 11: loss = 73126.226
Epoch 12: loss = 64566.765
Epoch 13: loss = 56291.697
Epoch 14: loss = 48602.214
Epoch 15: loss = 41766.919
Epoch 16: loss = 35227.251
Epoch 17: loss = 28793.978
Epoch 18: loss = 23571.833
Epoch 19: loss = 19378.066
Epoch 20: loss = 15906.685
Epoch 21: loss = 13496.213
Epoch 22: loss = 11081.047
Epoch 23: loss = 8396.176
Epoch 24: loss = 6265.476
Epoch 25: loss = 5030.006
Epoch 26: loss = 4458.622
Epoch 27: loss = 3454.725
Epoch 28: loss = 3398.630
Epoch 29: loss = 2676.799
Epoch 30: loss = 2458.720
Epoch 31: loss = 2180.038
Epoch 32: loss = 2169.137
Epoch 33: loss = 1988.366
Epoch 34: loss = 2075.997
Epoch 35: loss = 2019.921
Epoch 36: loss = 1799.336
Epoch 37: loss =

In [37]:
model.eval()
with torch.no_grad():
    for _ in range(5):
        i = np.random.randint(len(X))
        true = y[i].item()
        pred = model(X[i:i+1]).item()
        print(f"true: {true:6.1f} | pred: {pred:6.1f}")
model.train()


true:   -2.0 | pred:    5.3
true:    0.0 | pred:   -0.4
true:    4.0 | pred:    1.8
true: -1004.0 | pred: -940.2
true:   -2.0 | pred:    8.7


ValueNet(
  (net): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=768, out_features=256, bias=True)
    (2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [38]:
i, j = np.random.randint(len(X)), np.random.randint(len(X))

true_i, true_j = y[i].item(), y[j].item()

model.eval()
with torch.no_grad():
    pred_i = model(X[i:i+1]).item()
    pred_j = model(X[j:j+1]).item()

print("TRUE:", true_i, true_j)
print("PRED:", pred_i, pred_j)

TRUE: -3.0 0.0
PRED: 11.179096221923828 1.8453210592269897
